# RUN+Trip 장소 추천 PoC (Colab)

Notion "RUN+Trip - 장소 추천 모델 PoC" 기준. 위에서부터 셀을 차례로 실행하세요.

**준비 (처음 한 번)**: Google Drive `내 드라이브/RUNTrip/`에 아래 파일을 올려요.
```
RUNTrip/
├─ poc/   embed_kure.py, features.py, evaluate.py, model.py, losses.py, train.py
└─ data/  places.csv, queries.jsonl, labels.jsonl, split.json
```
- 팀원은 이 폴더를 공유받아 **내 드라이브에 바로가기 추가** 후 같은 경로로 쓰면 돼요.
- 런타임 유형은 GPU가 있으면 KURE 임베딩이 빨라요 (학습은 CPU로 충분).

## 1. Drive 연결 · 파일 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/RUNTrip'   # Drive에 올린 폴더 위치
%cd {ROOT}

import os
need = ['poc/embed_kure.py', 'poc/features.py', 'poc/evaluate.py', 'poc/model.py', 'poc/losses.py', 'poc/train.py',
        'data/places.csv', 'data/queries.jsonl', 'data/labels.jsonl', 'data/split.json']
missing = [p for p in need if not os.path.exists(p)]
print('누락 파일:', missing or '없음 ✅')

## 2. 설치

In [ ]:
!pip -q install sentence-transformers

## 3. KURE 임베딩 (팀에서 한 번만)
이미 `data/kure/`가 있으면 건너뛰어요. 첫 실행 시 모델(약 2.3GB)을 내려받아요.

In [ ]:
if os.path.exists('data/kure/place_vecs.npy'):
    print('KURE 임베딩 있음 → 건너뜀')
else:
    !python poc/embed_kure.py

## 4. 피처 생성 → `data/features.npz`

In [ ]:
!python poc/features.py

## 5. 규칙 기준선 (valid)

In [ ]:
!python poc/evaluate.py

## 6. 하이퍼파라미터 탐색 (valid)
`LOSS`를 자기 담당 후보로 바꾸고 실행해요. 조합 12개(lr 2 × dropout 2 × 시드 3) 결과가 `poc/runs/`에 저장돼요.

In [ ]:
LOSS = 'A'                 # 담당 후보: A(이민구) / B(노종원) / C(장현서)
LRS = [1e-3, 3e-4]
DROPOUTS = [0.1, 0.3]
SEEDS = [0, 1, 2]

In [ ]:
import itertools
for lr, do, seed in itertools.product(LRS, DROPOUTS, SEEDS):
    print(f'--- {LOSS} lr={lr} dropout={do} seed={seed}')
    !python poc/train.py --loss {LOSS} --lr {lr} --dropout {do} --seed {seed}

## 7. valid 결과 정리 (시드 3개 평균) · 후보별 최적 설정

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in glob.glob('poc/runs/*.json'):
    r = json.load(open(f, encoding='utf-8'))
    a = r['args']
    rows.append({'loss': a['loss'], 'lr': a['lr'], 'dropout': a['dropout'], 'seed': a['seed'],
                 **{f'valid_{k}': v for k, v in r['valid'].items()}})
df = pd.DataFrame(rows)
summary = (df.groupby(['loss', 'lr', 'dropout'])
             .agg(시드수=('seed', 'size'), **{c: (c, 'mean') for c in df.columns if c.startswith('valid_')})
             .round(4).reset_index())
display(summary)

best = summary.sort_values('valid_NDCG@5', ascending=False).groupby('loss').head(1)
print('후보별 최적 설정 (valid NDCG@5 기준)')
display(best)

## 8. test 평가 ⚠️ 튜닝이 끝난 뒤 한 번만
PoC 진행 규칙 3: test 셋은 마지막에 한 번만 평가해요. 7번에서 고른 최적 설정으로 시드 3개를 다시 학습하고 test를 평가해요.

In [ ]:
b = best[best['loss'] == LOSS].iloc[0]
for seed in SEEDS:
    !python poc/train.py --loss {LOSS} --lr {b['lr']} --dropout {b['dropout']} --seed {seed} --test

## 9. 최종 결과표 (Notion 6장에 기록)

In [ ]:
rows = []
for f in glob.glob('poc/runs/*.json'):
    r = json.load(open(f, encoding='utf-8'))
    if 'test' not in r:
        continue
    a = r['args']
    rows.append({'모델': a['loss'], 'lr': a['lr'], 'dropout': a['dropout'], 'seed': a['seed'],
                 **r['test'], '추론 시간(ms)': r['inference_ms']})
final = (pd.DataFrame(rows).groupby(['모델', 'lr', 'dropout'])
           .mean(numeric_only=True).drop(columns='seed').round(4))
display(final)

print('규칙 기준선 (test):')
!python poc/evaluate.py --test